# 09_binding_pocket_analysis

Notebook UI for UPO homolog pocket analysis using an LLM with binding, alignment, and optional reaction inputs.

## Python Path Setup
Ensure project-root imports work whether Jupyter starts from repo root or `notebooks/`.

In [1]:
from pathlib import Path
import os
import sys

cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
src_root = repo_root / "src"
if src_root.exists() and str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))


## Imports
Load helper functions for table loading, LLM analysis, output export, and thread persistence.

In [2]:
import importlib
import agentic_protein_design.steps.analyze_binding_pocket as bp
bp = importlib.reload(bp)
from agentic_protein_design.core.thread_context import load_optional_thread_context
from agentic_protein_design.core import apply_notebook_markdown_style, resolve_input_path

analyze_pocket_profiles = bp.analyze_pocket_profiles
default_user_inputs = bp.default_user_inputs
build_prompt_with_context = bp.build_prompt_with_context
generate_llm_pocket_analysis = bp.generate_llm_pocket_analysis
generate_llm_mutation_design_proposal = bp.generate_llm_mutation_design_proposal
run_llm_pocket_analysis_stages = bp.run_llm_pocket_analysis_stages
prompt_3 = bp.prompt_3
init_thread = bp.init_thread
load_input_tables = bp.load_input_tables
persist_thread_update = bp.persist_thread_update
save_llm_analysis = bp.save_llm_analysis
save_mutation_design_proposal = bp.save_mutation_design_proposal
save_rational_engineering_proposals = bp.save_rational_engineering_proposals
save_binding_outputs = bp.save_binding_outputs
setup_data_root = bp.setup_data_root
get_step_processed_dir = bp.get_step_processed_dir
REQUIRED_SUBFOLDERS = bp.REQUIRED_SUBFOLDERS

apply_notebook_markdown_style(font_size_px=14, line_height=1.4)


## User Inputs
Edit all run parameters here (single place): dataset root, thread selection, analysis options, model, and input paths.

In [3]:
root_key = "ECOHARVEST" # "examples"
existing_thread_key = 'binding_pocket_llm_analysis_lipases_b4e2b28e48164985a8e4515ff0b0d417' # "binding_pocket_llm_analysis_UPOs_59353c876ab140688b1c239a15aac24e"  # None

user_inputs = {
    "selected_positions": None, # [100, 103, 104, 107, 141, 222],
    "pairwise_comparisons":  [("RML", "TLL")], # [("CviUPO", "ET096")], # None
    "focus_question": (
        "Identify per-protein structural interpretations and cross-homolog patterns "
        "that could explain activity/property differences."
    ),
    "design_requirements": (
        "Backbone: RML. Goal: improve esterification of oleic acid with sucrose sugar in an environment containing water to enhance transport of the sugar."
        # "Backbone: ET096. Goal: improve peroxygenative mono-oxidation selectivity on S82 while retaining useful activity and limiting over-oxidation to Di-Ox. Prioritize conservative, mechanistically justified mutations and a first-round panel <= 12 variants."
    ),
    "literature_context_thread_key": "literature_review_lipases_b713603189544094bf0c3aea97730dd1", #"literature_review_UPOs_d762a72ec7f04bec9b66ccd3aac21b91",  # Optional: literature-review thread key
    "reaction_data_description": "",
    # "reaction_data_description": (
    #     "- Veratryl alcohol: peroxygenative\n"
    #     "- Naphthalene: peroxygenative\n"
    #     "- NBD: peroxygenative\n"
    #     "- ABTS: peroxidative\n"
    #     "- S82: mixed; Mono-Ox ~ peroxygenation-biased, Di-Ox ~ peroxidation-biased\n"
    #     "Use ratios (e.g. Mono-Ox : Di-Ox) to infer peroxygenation vs peroxidation balance."
    # ),
    "use_reaction_data": False, # True,
    "llm_model": "gpt-5.2",
    "llm_temperature": 0.2,
    "llm_max_rows_per_table": 300,
}

input_paths = {
    # Paths are relative to the data root from project_config.variables.address_dict[root_key].
    "binding_csv": "pdb/lipases/bindingpocket_analysis.csv", # "pdb/bindingpocket_analysis.csv",
    "alignment_csv": "msa/lipases/RML_TLL_ali_withDist_FILT.csv", # "msa/UPO_peroxygenation_ali_withDist_FILT.csv",
    "reaction_data_csv": "expdata/substrate_reaction_data.csv",
}

# Optional: reset analysis options from helper defaults
# user_inputs = default_user_inputs()


## Setup Runtime Context
Initialize data directories and active chat thread from the values above.

In [4]:
data_root, resolved_dirs = setup_data_root(root_key, REQUIRED_SUBFOLDERS)
step_processed_dir = get_step_processed_dir(resolved_dirs)
thread, threads_preview = init_thread(root_key, existing_thread_key)
thread_id = thread["thread_id"]
data_root, step_processed_dir, thread_id


(PosixPath('/Users/charmainechia/Documents/projects/ECOHARVEST'),
 PosixPath('/Users/charmainechia/Documents/projects/ECOHARVEST/processed/09_binding_pocket_analysis'),
 'b4e2b28e48164985a8e4515ff0b0d417')

## Load Input Tables
Load descriptor and alignment tables, and optional reaction data, from `input_paths`.

In [5]:
binding_csv = resolve_input_path(data_root, input_paths["binding_csv"])
alignment_csv = resolve_input_path(data_root, input_paths["alignment_csv"])
reaction_data_csv = None
if user_inputs.get("use_reaction_data", False) and input_paths.get("reaction_data_csv", "").strip():
    reaction_data_csv = resolve_input_path(data_root, input_paths["reaction_data_csv"])

pocket, ali, reaction_df = load_input_tables(binding_csv, alignment_csv, reaction_data_csv)
binding_csv, alignment_csv, reaction_data_csv, pocket.head(3), (None if reaction_df is None else reaction_df.head(3))


(PosixPath('/Users/charmainechia/Documents/projects/ECOHARVEST/pdb/lipases/bindingpocket_analysis.csv'),
 PosixPath('/Users/charmainechia/Documents/projects/ECOHARVEST/msa/lipases/RML_TLL_ali_withDist_FILT.csv'),
 None,
          struct_name      struct_name.1      struct_name.2  \
 0  RML_SucroseOleate  RML_SucroseOleate  RML_SucroseOleate   
 1  TLL_SucroseOleate  TLL_SucroseOleate  TLL_SucroseOleate   
 
    num_pocket_res_ali  num_pocket_res<8  reactive_center_distance  \
 0                  57                51                     6.385   
 1                  60                48                     5.974   
 
    median_dist_res_to_ligand_reactive_center  median_min_dist_res_to_ligand  \
 0                                     10.261                          5.004   
 1                                     10.356                          5.537   
 
    mean_min_dist_to_centroid (distal)  mean_min_dist_to_centroid (proximal)  \
 0                               8.856                 

## Structured Exports
Generate heuristic comparative tables and export CSVs to `processed/`.

In [6]:
selected_positions = user_inputs["selected_positions"]
interp_df, pattern_summary = analyze_pocket_profiles(pocket, ali, selected_positions)
out_interp, out_patterns = save_binding_outputs(interp_df, pattern_summary, step_processed_dir)


## LLM Pocket Analysis
Query the LLM client with the full prompt and input tables, then save markdown output.


In [7]:
# Run two-stage LLM analysis
stage_outputs = run_llm_pocket_analysis_stages(pocket, ali, reaction_df, user_inputs)
prompt_2_output = stage_outputs["prompt_2_output"]
llm_analysis = stage_outputs["combined_analysis"]

out_llm = save_llm_analysis(llm_analysis, step_processed_dir)

# Prompt 3 defaults (overwritten in the next cell)
mutation_design_text = ""
out_mutation_design = None
literature_context_thread_key = None

print(out_llm)


### Binding Pocket Analysis - Stage 1

<details><summary>Prompt</summary>

```text

Analyse the uploaded inputs for a set of proteins to interpret how binding-pocket structure relates to catalytic activity and selectivity. 
Consider how both the proximal (<6 Å from docked ligand) and distal (up to ~11 Å from binding pocket centroid) residues affect the binding pocket environment.

INPUTS
- binding_pocket_table: extracted binding-pocket properties (per protein), calculated separately over proximal and distal residue sets where available.
- pocket_alignment_table: filtered residue alignment of pocket-proximal positions.
- reaction_data (optional): enzyme activity data on substrates.

OBJECTIVE
For each protein, integrate structural descriptors with (optional) reaction data to infer mechanistic behavior and classify pocket phenotypes.

TASKS

1) For each protein:
   - Generate a punchy tagline.
   - Provide a concise 5-6 bullet summary addressing:
        (i) proximal electrostatics  
        (ii) proximal sterics  
        (iii) distal electrostatics  
        (iv) distal sterics / outer pocket size  
        (v) overall synthesis of pocket phenotype, integrating structural properties with catalytic implications:
            - Interpret how geometry and chemistry influence productive (peroxygenative) vs competing (peroxidative) pathways.
            - If reaction_data is provided, use it to support structure–function relationships.

   Use the following column groups:

   PROXIMAL ELECTROSTATICS
   - charged_fraction (proximal), polar_fraction (proximal)
   - kd_weighted (proximal), hw_weighted (proximal)
   - median_dist_res_to_ligand_reactive_center

   PROXIMAL STERICS
   - mean_volume (proximal), weighted_mean_volume (proximal)
   - volume_variance (proximal)
   - small_residue_frac (proximal), bulky_residue_frac (proximal)
   - median_min_dist_res_to_ligand
   - reactive_center_distance
   - num_pocket_res_lt6

   DISTAL ELECTROSTATICS
   - charged_fraction (distal), polar_fraction (distal)
   - kd_weighted (distal), hw_weighted (distal)

   DISTAL STERICS / OUTER POCKET SIZE
   - mean_dist_to_centroid
   - mean_min_dist_to_centroid
   - mean_dist_backbone_to_centroid
   - mean_volume (distal)
   - volume_variance (distal)
   - small_residue_frac (distal), bulky_residue_frac (distal)
   - num_pocket_res_ali

   If proximal/distal suffixes are not explicitly present, infer proximal/distal groupings from context and state your assumption briefly.

2) Comparative analysis requirements (do BOTH):
   A) Intra-protein variant analysis (MANDATORY when variants are present):
   - Detect proteins that share the same base protein identity but differ by variant/mutation labels.
   - For each such protein family, explicitly compare each variant against its WT/reference form (if WT/reference is present).
   - If WT is not explicitly labeled, infer the closest reference sequence in that family and state the assumption.
   - For each variant-vs-reference comparison, report which structural dimensions changed:
        (i) proximal electrostatics
        (ii) proximal sterics
        (iii) distal electrostatics
        (iv) distal sterics / outer pocket size
   - Provide a mechanistic rationale linking those differences to functional shifts.

   B) User-requested pairwise comparisons:
   - Pairwise comparisons requested: RML vs TLL
   - Perform each requested pairwise comparison in addition to section A.
   - Explicitly contrast which structural dimensions changed (prox electrostatics, prox sterics, distal electrostatics, distal sterics).
   - Provide a mechanistic rationale for functional shifts.

3) Distill cross-protein trends or clusters (“pocket phenotypes”):
   - Identify recurring structural archetypes (e.g., tight/polar pose-locking vs open/hydrophobic permissive).
   - Link clusters to turnover vs selectivity trade-offs.

OUTPUT STYLE
- Clear, human-interpretable, mechanistically grounded.
- Emphasize intuition over raw numbers.
- Keep summaries compact and comparative.


REACTION CONTEXT (OPTIONAL)
If reaction_data is provided, use it to support structure–function reasoning.

REACTION_DATA_STATUS: not provided.
```
</details>

#### Response

(Stage 1 output included in compact combined view below.)

### Binding Pocket Analysis - Stage 2

...

#### Response

(Stage 2 output included in compact combined view below.)

### Combined Pocket Analysis

## Stage 1: Global Pocket Phenotypes

Assumptions/notes on the inputs  
- Your `binding_pocket_table` already contains **explicit proximal vs distal columns** for most descriptors (e.g., `mean_volume (proximal)` and `mean_volume (distal)`), so I treat those as the <6 Å vs ~6–11 Å residue sets, respectively.  
- `num_pocket_res<8` appears to be a *near-pocket count* but not strictly “<6 Å”; since `num_pocket_res_lt6` is not present, I use `num_pocket_res<8` as a **proxy for proximal packing density** and state that explicitly where relevant.  
- No reaction_data were provided, so catalytic/selectivity inferences are **mechanistic hypotheses** grounded in pocket physics + known lipase behavior (RML/TLL lid lipases; sucrose acylation often limited by sugar accommodation).

---

## Per-protein interpretations

### 1) **RML_SucroseOleate**
**Tagline:** *“Balanced, slightly polar pocket with a roomy outer vestibule—good for binding, less for pose-locking.”*

- **(i) Proximal electrostatics**
  - Moderate **charged_fraction ~0.182** and high **polar_fraction ~0.455**: a chemically “wettable” near-field that can H-bond to sucrose hydroxyls.
  - **kd_weighted (prox) = -0.158** (more negative) suggests the proximal shell trends **more hydrophilic/polar** than TLL’s proximal shell in this dataset.
  - **median_dist_res_to_ligand_reactive_center ~10.26 Å** indicates many “pocket residues” are not tightly focused around the reactive center—consistent with a broader cavity where only a subset truly steers chemistry.

- **(ii) Proximal sterics**
  - **median_min_dist_res_to_ligand ~5.00 Å** (closer than TLL) suggests **tighter local contact** to the docked ligand overall.
  - **mean_volume (prox) ~102.9 Å³; weighted_mean_volume (prox) ~100.9 Å³**: moderate sidechain bulk; not an extremely tight, small-residue-lined pocket.
  - **bulky_residue_frac (prox) ~0.341** with **small_residue_frac (prox) ~0.25**: mixed lining → tends to allow multiple microposes rather than a single “keyed” pose.
  - **reactive_center_distance ~6.39 Å**: reactive center is not extremely close to the catalytic machinery/idealized reactive geometry (relative to TLL here), which can reduce probability of a highly productive near-attack conformation.

- **(iii) Distal electrostatics**
  - Distal shell is similarly polar (**polar_fraction ~0.456**) but slightly less charged (**charged_fraction ~0.175**).
  - **kd_weighted (dist) ~0.021** shifts toward more hydrophobic/neutral compared to proximal, implying a **polarity gradient**: polar near-field with a more permissive outer region.

- **(iv) Distal sterics / outer pocket size**
  - Outer pocket distances are **moderate**: `mean_dist_to_centroid (distal) ~10.69 Å`, `mean_min_dist_to_centroid (distal) ~8.86 Å`.
  - Distal bulk is moderate (**mean_volume (dist) ~102.5 Å³; bulky_frac (dist) ~0.351**) with substantial heterogeneity (**volume_variance (dist) ~909**), consistent with a **textured vestibule** that can accommodate different ligand orientations.

- **(v) Pocket phenotype → catalytic implications (peroxygenative vs peroxidative competition)**
  - **Phenotype:** “balanced polar/hydrophobic, moderately open, not strongly pose-locking.”
  - Mechanistically, this kind of pocket tends to **admit sucrose** (polar contacts available) but may **struggle to enforce a single productive alignment** of the acceptor OH relative to the acyl-enzyme (or reactive intermediate), increasing the chance of **non-productive binding** and/or alternative microstates.
  - In a peracid/peroxide context (if that’s your competing-pathway framing), a **less constrained reactive geometry** generally increases the probability of **off-pathway peroxide reactions** (peroxidative) because the pocket does not “funnel” the reactive species into one dominant near-attack trajectory.

---

### 2) **TLL_SucroseOleate**
**Tagline:** *“More hydrophobic and bulkier near the ligand—built to gate and steer, not to solvate.”*

- **(i) Proximal electrostatics**
  - **charged_fraction (prox) ~0.182** (same as RML) but **polar_fraction (prox) ~0.409** (lower than RML): proximal environment is **less H-bond rich**.
  - **kd_weighted (prox) = -0.064** is less negative than RML → proximal shell is **less polar/more hydrophobic-leaning** than RML by this metric.
  - **median_dist_res_to_ligand_reactive_center ~10.36 Å** similar to RML: again suggests many annotated pocket residues are not tightly centered on the reactive center.

- **(ii) Proximal sterics**
  - **median_min_dist_res_to_ligand ~5.54 Å** (larger than RML): fewer very close contacts overall, but…
  - The lining is **bulkier**: **mean_volume (prox) ~107.9 Å³**, **bulky_residue_frac (prox) ~0.477** (substantially higher than RML’s 0.341), and **volume_variance (prox) ~1112** (higher).
  - Interpretation: TLL’s proximal pocket is more like a **sculpted, bulky “funnel”**—not necessarily closer everywhere, but more capable of **steric steering** and excluding certain poses.

- **(iii) Distal electrostatics**
  - Distal shell has **higher charged_fraction ~0.20** but **lower polar_fraction ~0.40** than RML.
  - Net effect: distal region may present **more discrete charge points** (salt-bridge opportunities) but fewer distributed H-bond donors/acceptors—often consistent with **specific anchoring sites** rather than general sugar solvation.

- **(iv) Distal sterics / outer pocket size**
  - Distal distances are **larger** than RML: `mean_dist_to_centroid (distal) ~11.29 Å` and `mean_min_dist_to_centroid (distal) ~9.38 Å` → a **bigger outer vestibule**.
  - Distal bulk is slightly higher (**mean_volume (dist) ~104.3 Å³; bulky_frac (dist) ~0.417**) with high heterogeneity (**volume_variance (dist) ~1060**): suggests an **expanded but structured outer region** that can host the sugar while the acyl chain occupies a hydrophobic groove.

- **(v) Pocket phenotype → catalytic implications (peroxygenative vs peroxidative competition)**
  - **Phenotype:** “outer-vestibule roomy, inner pocket sterically directive and more hydrophobic.”
  - This combination often supports **selectivity**: the distal vestibule can “park” a bulky polar acceptor (sucrose) while the proximal bulky/hydrophobic features **bias which hydroxyl can approach** the reactive center (consistent with literature tendencies of TLL toward **regioselective monoacylation** on sucrose).
  - In the productive-vs-competing framing: stronger steric steering near the reactive center generally **reduces off-pathway chemistry** by limiting reactive species orientations—i.e., it should *favor productive (peroxygenative-like) trajectories* over diffuse peroxidative side reactions, assuming the reactive intermediate is generated in-pocket.

---

## 2) Comparative analysis

### A) Intra-protein variant analysis
- **No variant families detected** in the provided structures: only `RML_SucroseOleate` and `TLL_SucroseOleate`, which are different homologs rather than variants of the same base protein.  
- Therefore, **no WT-vs-variant comparisons** are possible from these inputs.

### B) Requested pairwise comparison: **RML vs TLL**

**Proximal electrostatics**
- **RML is more polar in the near field**: polar_fraction 0.455 (RML) vs 0.409 (TLL); kd_weighted more negative (-0.158 vs -0.064).
- Mechanistic implication: RML likely provides **better H-bonding “landing”** for sucrose near the catalytic region, but that can also stabilize **multiple non-productive sugar poses** unless sterics enforce one.

**Proximal sterics**
- **TLL is markedly bulkier and more heterogeneous proximally**: bulky_residue_frac 0.477 vs 0.341; mean_volume 107.9 vs 102.9; volume_variance 1112 vs 881.
- RML has **closer overall contacts** to the docked ligand (median_min_dist 5.00 vs 5.54), but TLL has **more steric shaping power**.
- Mechanistic implication: TLL’s bulky proximal shell is better suited to **pose selection/regioselectivity** (exclude wrong hydroxyl approaches), whereas RML’s mixed/less-bulky lining is more permissive.

**Distal electrostatics**
- RML distal is **more polar overall** (polar_fraction 0.456 vs 0.400), while TLL distal is **more charged** (0.20 vs 0.175).
- Mechanistic implication: RML distal region may better support **general sugar accommodation**; TLL may rely on **specific charge anchors** (fewer but stronger interaction points).

**Distal sterics / outer pocket size**
- **TLL has a larger outer vestibule** (mean_dist_to_centroid distal 11.29 vs 10.69; mean_min_dist_to_centroid distal 9.38 vs 8.86) and is also bulkier distally (bulky_frac 0.417 vs 0.351).
- Mechanistic implication: TLL can better **host bulky sucrose** in the outer region while still enforcing a constrained approach near the reactive center—often a recipe for **monoacylation selectivity** (park-and-react geometry).

**Pocket-alignment “where the differences likely come from” (proximal positions)**
- Several aligned substitutions increase TLL polarity/charge or bulk at specific sites (examples from your table):
  - RML 83 **S** → TLL 84 **R** (adds a cationic, bulky sidechain near-pocket)
  - RML 91 **D** → TLL 92 **N** (removes negative charge)
  - RML 90 **A** → TLL 91 **G** (local flexibility change)
  - RML 259 **S** → TLL 260 **W** (adds bulky aromatic)
  - RML 264 **N** → TLL 264 **L** (more hydrophobic)
- Net: TLL substitutions skew toward **bulkier/hydrophobic shaping** plus **strategic charge placement**, consistent with the global descriptors.

---

## 3) Cross-protein “pocket phenotypes” (clusters) from this small set

With only two homologs, the clustering is necessarily coarse, but two clear archetypes emerge:

1) **“Polar-permissive pocket” (RML-like)**
- Higher proximal + distal polar fractions; more hydrophilic proximal kd_weighted.
- Moderate steric shaping (lower bulky fraction).
- Trade-off hypothesis: **better acceptance of polar substrates** (sucrose entry/retention) but **lower intrinsic pose discrimination**, which can reduce regioselectivity and increase non-productive binding.

2) **“Vestibule-parking + steric steering” (TLL-like)**
- Larger distal vestibule + higher bulky fraction proximally and distally.
- Slightly more hydrophobic proximal environment with discrete charges.
- Trade-off hypothesis: **higher selectivity (pose filtering/regioselectivity)**, potentially at the cost of requiring the substrate to find a narrower set of productive approaches (which can reduce turnover if docking/entry is limiting).

If you later add reaction_data (conversion, mono/di ratio, regioisomer distribution), we can test these hypotheses directly—especially whether TLL’s “steering” correlates with monoacylation enrichment and whether RML’s polarity correlates with higher binding/occupancy but broader product distributions.

## Stage 2: Residue-Level Mechanistic Drivers

## 1) Key variable pocket positions → residue-level mechanistic hypotheses (RML vs TLL)

Below I focus on **positions that (a) differ in identity, and (b) are plausibly causal for the global phenotypes** you summarized: **RML = more polar/permissive near-field**, **TLL = bulkier/more hydrophobic steric steering + discrete charge anchors**, with **TLL having a larger distal vestibule**.

> Note: you requested variant-vs-base comparisons; **none exist in these inputs** (only two homologs). I therefore contrast **RML vs TLL** at each position.

---

### A. Proximal / near-field drivers (most likely to impact sucrose OH approach & pose selection)

#### **RML 83 Ser ↔ TLL 84 Arg** (min dist ~3.5–3.6 Å)
- **Substitution class**
  - **Electrostatic:** neutral → **cationic**
  - **Steric:** small → **bulky**
  - **Polarity:** polar uncharged → strongly polar/charged
- **Mechanistic consequence**
  - In **TLL**, Arg introduces a **localized positive electrostatic anchor** and a **steric “post”** near the ligand. This matches your phenotype of **“discrete charge points” + steric steering**.
  - Likely effects:
    - **Bias sucrose orientation** by stabilizing specific hydroxyl/oxygen patterns (or phosphate/sulfate if present; here likely sucrose OH network).
    - **Reduce microstate degeneracy** (fewer non-productive sugar poses) by steric exclusion → consistent with **TLL regioselective monoacylation tendency**.
    - Potential downside: could **penalize entry/retention** of sucrose if it creates an overly specific H-bond/salt-bridge geometry (especially in low-water organics where charge desolvation is costly).
  - In **RML**, Ser keeps the region **H-bond capable but permissive**, consistent with your “polar-permissive, less pose-locking” description.

#### **RML 91 Asp ↔ TLL 92 Asn** (min dist ~2.37 vs 3.30 Å; very close in RML)
- **Substitution class**
  - **Electrostatic:** **negative → neutral**
  - **Polarity:** charged polar → polar amide (still H-bonding)
- **Mechanistic consequence**
  - In **RML**, Asp at very close contact distance can create a **strong, directional electrostatic field** (and potentially a persistent H-bond acceptor) that can:
    - **Over-stabilize multiple sucrose OH binding modes** (many hydroxyls can satisfy Asp), increasing **non-productive binding**—aligns with your “good for binding, less for pose-locking.”
    - Potentially **repel** negatively polarized groups and **attract** hydroxyl protons, altering which OH is presented toward the acyl-enzyme.
  - In **TLL**, Asn removes the formal charge while retaining H-bonding, which likely:
    - **Reduces “sticky” nonspecific electrostatic trapping** of sucrose near the reactive region.
    - Supports your observation that TLL is **less H-bond rich overall** proximally, but can still make **specific** polar contacts.

#### **RML 265 Thr ↔ TLL 265 Ile** (min dist ~2.91 vs 3.97 Å)
- **Substitution class**
  - **Polarity:** polar → **hydrophobic**
  - **Steric:** modest increase in hydrophobic bulk/shape
- **Mechanistic consequence**
  - **TLL Ile** strengthens a **hydrophobic wall** near the ligand, consistent with your “more hydrophobic/bulkier near the ligand—built to gate and steer.”
  - Likely effects:
    - **Favors acyl-chain packing** and can **push sucrose away** from that face, narrowing approach trajectories (pose filtering).
  - **RML Thr** provides an extra **H-bond donor/acceptor** close to ligand, reinforcing the **polar landing pad** behavior and potentially increasing alternative sucrose microposes.

---

### B. Distal / vestibule-shaping drivers (likely to affect sucrose parking, access channel geometry, and “outer vestibule” phenotype)

#### **RML 259 Ser ↔ TLL 260 Trp** (min dist ~7.6–7.7 Å; distal but pocket-facing)
- **Substitution class**
  - **Steric:** small → **very bulky aromatic**
  - **Polarity:** polar → largely hydrophobic (with indole NH)
- **Mechanistic consequence**
  - **TLL Trp** is a classic **steric gate / wall-former**:
    - Can **sculpt the vestibule** and create a **defined “parking surface”** (π/CH contacts) for sugar rings.
    - Can **reduce solvent exposure** and enforce a more **channeled approach** from vestibule → reactive center, consistent with your “vestibule-parking + steric steering” model.
  - **RML Ser** keeps this region **open and wettable**, consistent with a **textured but permissive vestibule** and higher distal polar fraction.

#### **RML 264 Asn ↔ TLL 264 Leu** (min dist ~6.18 vs 7.88 Å)
- **Substitution class**
  - **Polarity:** polar amide → **hydrophobic**
  - **Steric:** similar size but different shape/packing; Leu increases hydrophobic surface continuity
- **Mechanistic consequence**
  - **TLL Leu** supports a **more hydrophobic vestibule wall**, consistent with lower polar_fraction(distal) but slightly higher bulky_frac(distal).
  - Likely shifts sucrose behavior from “solvated/retained by many H-bonds” (RML-like) to “parked by shape + a few anchors” (TLL-like).

---

### C. Additional variable positions (probable secondary effects)

#### **RML 90 Ala ↔ TLL 91 Gly** (min dist ~5.6–6.5 Å)
- **Substitution class**
  - **Steric/flexibility:** Ala → **Gly increases backbone flexibility**
- **Mechanistic consequence**
  - Could subtly alter **local loop/turn mobility** near the pocket, affecting **gating dynamics** (important for lid lipases), but likely **secondary** unless this sits on a key shaping loop.

#### **RML 93 Thr ↔ TLL 94 Asn** (min dist ~5.0–6.7 Å)
- **Substitution class**
  - **Polarity shift:** Thr (OH) → Asn (amide); both polar, Asn more H-bond acceptor-rich
- **Mechanistic consequence**
  - Fine-tunes H-bond patterning; likely **modulatory** compared to the Asp/Arg changes above.

#### **RML 174 Gln ↔ TLL 171 Tyr** (distal; min dist ~7.5–7.8 Å)
- **Substitution class**
  - **Polarity/π:** Gln polar → Tyr aromatic polar (phenolic OH)
  - **Steric:** moderate increase in rigid bulk
- **Mechanistic consequence**
  - Tyr can provide a **rigid aromatic platform** for sugar ring contacts; may contribute to **structured vestibule** in TLL.

#### **RML 176 Gln ↔ TLL 173 Ala** (distal; min dist ~6.6–7.0 Å)
- **Substitution class**
  - **Polarity:** polar → **nonpolar**
  - **Steric:** smaller
- **Mechanistic consequence**
  - Removes a distal H-bond site in TLL, consistent with **lower distal polar_fraction** and more reliance on **shape/limited anchors**.

#### **RML 207 His ↔ TLL 205 Arg** (distal; min dist ~7.0–7.5 Å)
- **Substitution class**
  - **Electrostatic:** potentially neutral/weakly cationic (His) → **strong cation (Arg)**
  - **Steric:** larger
- **Mechanistic consequence**
  - Adds another **discrete positive charge point** in TLL distal shell, consistent with your “more charged distally” observation; could help **capture/position sucrose** in the vestibule without making the whole region highly polar.

#### **RML 215 Phe ↔ TLL 213 Tyr** (distal; min dist ~6.8 Å)
- **Substitution class**
  - **Polarity:** hydrophobic aromatic → aromatic with OH (slightly more polar)
- **Mechanistic consequence**
  - Small tuning of distal H-bonding; likely **minor** relative to Trp/Leu/Arg changes.

#### **RML 254 Val ↔ TLL 255 Ile** (distal; min dist ~5.8 vs 3.5 Å)
- **Substitution class**
  - **Steric:** Val → Ile (slightly bulkier)
- **Mechanistic consequence**
  - Could contribute to **tighter hydrophobic packing** in TLL at that spot; likely **secondary**.

#### **RML 267 Leu ↔ TLL 267 Thr** (distal; min dist ~3.8 vs 5.1 Å)
- **Substitution class**
  - **Polarity:** hydrophobic → polar
- **Mechanistic consequence**
  - This is one of the few changes that could make TLL *more* polar locally; may serve as a **specific H-bond “handle”** amid an otherwise hydrophobic steering surface.

---

## 2) Ranked residue list (mechanistic drivers vs modulators vs likely neutral)

### High-confidence mechanistic driver residues (most causal for your global phenotypes)
1. **RML 83 Ser ↔ TLL 84 Arg** — adds **bulky cationic anchor** near pocket; strong steric + electrostatic steering (fits TLL selectivity phenotype).
2. **RML 91 Asp ↔ TLL 92 Asn** — removes **negative charge** at very close contact; likely major contributor to **RML higher near-field polarity/permissiveness** vs TLL.
3. **RML 259 Ser ↔ TLL 260 Trp** — major **vestibule sculpting/gating** via bulky aromatic; supports “structured vestibule + steering” in TLL.
4. **RML 264 Asn ↔ TLL 264 Leu** — shifts distal wall from **polar to hydrophobic**, reinforcing TLL’s less polar vestibule.

### Secondary modulators (context-dependent; tune rather than define phenotype)
- **RML 265 Thr ↔ TLL 265 Ile** — local hydrophobic wall vs H-bond site near ligand.
- **RML 207 His ↔ TLL 205 Arg** — adds distal discrete positive charge (anchoring).
- **RML 176 Gln ↔ TLL 173 Ala** and **RML 174 Gln ↔ TLL 171 Tyr** — reshape distal H-bond availability / aromatic packing.
- **RML 267 Leu ↔ TLL 267 Thr** — introduces a polar “handle” in TLL; may compensate for other hydrophobization.

### Likely neutral/background (small effects or indirect unless coupled)
- **RML 90 Ala ↔ TLL 91 Gly** — flexibility tweak; probably minor alone.
- **RML 93 Thr ↔ TLL 94 Asn** — polar↔polar swap; subtle.
- **RML 215 Phe ↔ TLL 213 Tyr** — minor polarity increase.
- **RML 254 Val ↔ TLL 255 Ile** — conservative hydrophobic packing change.

---

If you share **which residues are proximal (<6 Å) vs distal (6–11 Å) in your exact definition** (or provide the full pocket table with proximal/distal labels per residue), I can tighten the causal chain further (e.g., explicitly mapping which of these sit in the alcohol-binding region vs acyl groove vs lid-adjacent gate).

/Users/charmainechia/Documents/projects/ECOHARVEST/processed/09_binding_pocket_analysis/binding_pocket_llm_analysis.md


## LLM Backbone Engineering Proposal
Use Stage-2 residue-level drivers plus optional literature-thread context to propose mutation designs under user requirements.

In [8]:
design_requirements = str(user_inputs.get("design_requirements", "")).strip()
literature_context_thread_key = str(user_inputs.get("literature_context_thread_key", "")).strip() or None

context_result = load_optional_thread_context(
    literature_context_thread_key,
    include_referenced_files=False,
    max_chars_per_file=40000,
    on_missing="warn",
    json_artifact_names=["engineering_strategy"],
)
literature_context_bundle = context_result.get("context_bundle")
engineering_strategy = ((literature_context_bundle or {}).get("referenced_json_objects") or {}).get("engineering_strategy", {})

mutation_design_outputs = generate_llm_mutation_design_proposal(
    prompt_2_output=prompt_2_output,
    design_requirements=design_requirements,
    user_inputs=user_inputs,
    engineering_strategy=engineering_strategy,
)
mutation_design_text = str(mutation_design_outputs.get("prompt_3_output_text", ""))
proposals_df = mutation_design_outputs.get("proposals_df")
out_mutation_design = save_mutation_design_proposal(mutation_design_text, step_processed_dir)
out_rational_proposals = save_rational_engineering_proposals(proposals_df, step_processed_dir)

{"mutation_design_path": str(out_mutation_design), "rational_engineering_proposals_path": str(out_rational_proposals), "n_rows": 0 if proposals_df is None else int(len(proposals_df))}, proposals_df.head(20) if proposals_df is not None else proposals_df


### Binding Pocket Mutation Design Proposal

<details><summary>Prompt</summary>

```text

You are designing enzyme variants for rational engineering.

You are given:
1) prompt_2_output: residue-level mechanistic analysis of binding-pocket drivers.
2) engineering_strategy (optional): structured literature-derived strategy JSON.
3) design_requirements: user-provided requirements including:
   - target backbone protein to engineer
   - engineering aims (activity/selectivity/stability/pathway bias)
   - constraints (allowed positions, mutation budget, excluded residues/motifs, expression or assay limits)

TASK
Generate a concrete mutation design proposal grounded primarily in prompt_2_output and supported by engineering_strategy when relevant.

OUTPUT FORMAT
1) Design Intent
   - State backbone protein and explicit engineering objective.

2) Proposed Mutations (ranked)
   - Provide 5-10 proposals total.
   - Include both:
     - specific substitutions (e.g., F88L), and
     - optional position-level exploration suggestions (e.g., site-saturation at position 158 with a small focused set).
   - For each proposal provide:
     - rank
     - proposal (mutation or position-set)
     - rationale linked to prompt_2 mechanistic driver(s)
     - which engineering hypothesis from INPUT_DATA_JSON (added below under this prompt) this targets, if relevant
     - expected directional effect on function
     - risk/tradeoff
     - confidence (high/medium/low)

3) Minimal Experimental Plan
   - Suggest a compact first-round panel (6-12 variants max), prioritizing high-information designs.
   - Include a short assay/readout plan aligned with the objective.

4) Rejected Alternatives
   - Briefly list 3-5 plausible but lower-priority options and why they were deprioritized.

RULES
- Do not invent residue numbering outside the provided context.
- Keep causal links explicit from residue-level mechanism -> mutation -> expected phenotype.
- If engineering_strategy conflicts with prompt_2_output, state the conflict and choose a conservative design.
- Prefer practical, testable proposals over speculative broad recommendations.

```
</details>

#### Response

(Full mutation proposal shown below in compact view.)

### Mutation Design Proposal

## 1) Design Intent
- **Backbone protein:** RML lipase
- **Objective:** **Increase esterification of oleic acid with sucrose in a water-containing environment** by shifting RML’s pocket from “polar-permissive/many microposes” toward **more pose-locking steric steering + discrete anchoring**, while retaining enough polar handling to keep sucrose engaged under aqueous conditions.

---

## 2) Proposed Mutations (ranked)

### 1) **S83R**
- **Rationale (mechanistic driver):** prompt_2 identifies **RML 83 Ser ↔ TLL 84 Arg** as a top proximal driver. Introducing **Arg** adds a **bulky cationic anchor + steric post** near sucrose (3.5–3.6 Å), expected to **reduce non-productive sucrose poses** and bias a productive OH presentation.
- **Targets hypothesis:** “TLL-like discrete charge points + steric steering improves regioselective/pose-locked binding.”
- **Expected effect:** ↑ productive sucrose orientation → ↑ esterification rate/extent (especially if RML currently binds sucrose too degenerately).
- **Risk/tradeoff:** In water-containing media, Arg is fine, but in lower-water microenvironments it can be costly to desolvate; may also **over-constrain** binding and reduce turnover if geometry mismatches.
- **Confidence:** **High**

### 2) **D91N**
- **Rationale:** **RML 91 Asp ↔ TLL 92 Asn** is very close contact in RML (~2.37 Å). Removing the **formal negative charge** should reduce “sticky” nonspecific electrostatic trapping of multiple sucrose OH microstates while keeping H-bonding via Asn.
- **Targets hypothesis:** “Reducing near-field charge decreases nonproductive binding and improves catalytic pose selection.”
- **Expected effect:** ↑ catalytic efficiency (less nonproductive binding), potentially ↑ esterification in water by avoiding overly strong/incorrect OH capture.
- **Risk/tradeoff:** Asp might currently help recruit/retain sucrose; neutralizing could reduce apparent binding at low sucrose.
- **Confidence:** **High**

### 3) **S83R + D91N (double)**
- **Rationale:** Combines the two strongest proximal drivers: **add a discrete positive anchor (Arg)** while **removing a potentially promiscuous negative trap (Asp→Asn)**. Mechanistically, this should convert RML’s near-field from “many H-bond solutions” to “fewer, more directed solutions.”
- **Targets hypothesis:** “Near-field electrostatic re-patterning is sufficient to shift RML toward TLL-like pose-locking.”
- **Expected effect:** ↑ esterification and potentially ↑ monoacylation bias (if pose-locking improves).
- **Risk/tradeoff:** Could overspecify sucrose pose and reduce turnover if the productive pose is not the one stabilized.
- **Confidence:** **Medium-High**

### 4) **T265I**
- **Rationale:** **RML 265 Thr ↔ TLL 265 Ile** is near-field (~2.9 Å in RML). Thr provides an H-bond site; Ile creates a **hydrophobic wall** that can **steer sucrose away from that face** and improve acyl-chain packing/organization for oleate.
- **Targets hypothesis:** “Hydrophobizing a near-field wall improves steric steering and acyl-chain accommodation.”
- **Expected effect:** ↑ esterification (better oleate positioning; fewer sucrose microposes).
- **Risk/tradeoff:** Might reduce sucrose residence time in water (loss of a polar contact) and/or reduce activity if Thr participates in a beneficial H-bond network.
- **Confidence:** **Medium**

### 5) **N264L**
- **Rationale:** **RML 264 Asn ↔ TLL 264 Leu** is a distal/vestibule-shaping driver. Leu increases **hydrophobic surface continuity** in the vestibule, pushing behavior toward “park by shape + limited anchors” rather than many distal H-bonds.
- **Targets hypothesis:** “Vestibule hydrophobization reduces solvent-wet, nonproductive parking and improves channeled approach.”
- **Expected effect:** Potential ↑ effective delivery of sucrose into productive near-field pose; may also help oleate access/packing.
- **Risk/tradeoff:** In water-containing media, too much hydrophobization can reduce sucrose capture/partitioning into the pocket.
- **Confidence:** **Medium**

### 6) **S259W**
- **Rationale:** **RML 259 Ser ↔ TLL 260 Trp** is a major distal vestibule sculptor. Trp can act as a **steric gate/wall** and provide **π/CH contacts** for sugar rings, potentially creating a more defined “parking surface” and channeled entry.
- **Targets hypothesis:** “Aromatic gating in the vestibule improves productive approach trajectories.”
- **Expected effect:** ↑ pose filtering; possibly ↑ monoacylation selectivity and ↑ esterification efficiency if sucrose is currently too mobile.
- **Risk/tradeoff:** High steric risk—Trp could **over-occlude** the vestibule and reduce substrate access, especially for bulky sucrose in water.
- **Confidence:** **Medium-Low** (high impact but higher failure risk)

### 7) **H207R**
- **Rationale:** **RML 207 His ↔ TLL 205 Arg** adds a **distal discrete positive charge point** (vestibule shell). In water-containing media, this may help **capture/hold sucrose** without making the whole pocket highly polar.
- **Targets hypothesis:** “Discrete distal anchoring improves sucrose recruitment/retention under aqueous conditions.”
- **Expected effect:** ↑ apparent binding/occupancy of sucrose → could increase esterification rate if binding is limiting.
- **Risk/tradeoff:** Could increase nonproductive binding if it anchors sucrose in a nonproductive vestibule pose; may alter pH dependence.
- **Confidence:** **Medium**

### 8) **Focused exploration at 83 (small set): S83R / S83K / S83H**
- **Rationale:** Position 83 is a top proximal driver. If Arg is too strong/too bulky, Lys or His may provide a **tunable cationic anchor** with different geometry and desolvation cost.
- **Targets hypothesis:** “A cationic post at 83 is beneficial, but optimal strength/geometry matters in water.”
- **Expected effect:** Identify best balance of pose-locking vs turnover.
- **Risk/tradeoff:** Requires a mini-panel; His may be partially protonated depending on pH.
- **Confidence:** **Medium**

---

## 3) Minimal Experimental Plan (first round: 10 variants)
High-information panel emphasizing the strongest mechanistic drivers plus one “vestibule gate” test:

1. WT RML  
2. **S83R**  
3. **D91N**  
4. **S83R/D91N**  
5. **T265I**  
6. **S83R/T265I** (tests synergy: anchor + hydrophobic steering)  
7. **N264L**  
8. **S259W** (single high-impact vestibule gate test)  
9. **H207R**  
10. **D91N/T265I** (tests whether charge removal + hydrophobic wall is sufficient without Arg)

### Assay/readout plan (aligned to esterification in water-containing media)
- **Primary reaction:** sucrose + oleic acid → sucrose oleate(s)
- **Quantification:** HPLC (or LC-MS) to measure:
  - **Total ester formation** (conversion/yield)
  - **Product distribution** (mono-/di-/poly-oleate; if resolvable)
- **Key conditions to include (small matrix):**
  - Fixed enzyme loading; time course (e.g., 0–24 h) to extract initial rates + endpoints
  - At least two water activities (e.g., “low water” vs “higher water” within your transport-relevant range) to see which variants retain activity when water competes.
- **Secondary checks (quick triage):**
  - Residual hydrolytic activity on a simple ester (to ensure enzyme is active/folded)
  - Expression/solubility screen (SDS-PAGE activity band or crude lysate activity)

---

## 4) Rejected Alternatives (lower priority)
1. **A90G** (RML 90 Ala ↔ TLL 91 Gly)  
   - Deprioritized: likely subtle loop flexibility effect; hard to predict benefit without dynamics data.

2. **T93N** (RML 93 Thr ↔ TLL 94 Asn)  
   - Deprioritized: polar↔polar swap; expected to be modulatory and lower impact than 83/91/259/264/265.

3. **F215Y** (RML 215 Phe ↔ TLL 213 Tyr)  
   - Deprioritized: minor distal polarity tweak; unlikely to move esterification strongly.

4. **V254I** (RML 254 Val ↔ TLL 255 Ile)  
   - Deprioritized: conservative hydrophobic packing change; secondary at best.

5. **Large distal rewiring (e.g., Q174Y and/or Q176A)**  
   - Deprioritized for round 1: could help vestibule structuring, but higher uncertainty and more likely to perturb folding/local packing without clear first-order linkage to the sucrose reactive pose.

If you can share whether your main limitation is **(i) sucrose binding/occupancy in water** vs **(ii) productive pose selection/regioselectivity**, I can reorder the panel (e.g., prioritize H207R vs S259W/N264L) and tighten the expected product-profile predictions.

({'mutation_design_path': '/Users/charmainechia/Documents/projects/ECOHARVEST/processed/09_binding_pocket_analysis/binding_pocket_mutation_design.md',
  'rational_engineering_proposals_path': '/Users/charmainechia/Documents/projects/ECOHARVEST/processed/09_binding_pocket_analysis/rational_engineering_proposals.csv',
  'n_rows': 15},
                                                mutant  \
 0                                                S83R   
 1                                                D91N   
 2                                         S83R + D91N   
 3                                               T265I   
 4                                               N264L   
 5                                               S259W   
 6                                               H207R   
 7   Focused library at position 83: S83R / S83K / ...   
 8                                          S83R/T265I   
 9                                          D91N/T265I   
 10                      A9

 ## Save Thread Update
Run this final cell to append run metadata and prompt context to `chats/<llm_process_tag>_<thread_id>.json`.

In [16]:
persist_thread_update(
    root_key=root_key,
    thread_id=thread_id,
    user_inputs=user_inputs,
    input_paths=input_paths,
    selected_positions=selected_positions,
    reaction_df=reaction_df,
    out_interp=out_interp,
    out_patterns=out_patterns,
    llm_analysis_path=out_llm,
    llm_analysis_text=llm_analysis,
    mutation_design_path=out_mutation_design,
    mutation_design_text=mutation_design_text,
    rational_engineering_proposals_path=out_rational_proposals,
    rational_engineering_proposals_rows=0 if proposals_df is None else int(len(proposals_df)),
    literature_context_thread_key=literature_context_thread_key,
    llm_model=str(user_inputs.get("llm_model", "")),
)


'2026-04-03T16:32:19.012873+00:00'